# E3 · Cross-attention fusion head (valence-arousal + mismatch)

Trains `pipeline.nets.CrossAttentionFusionHead` (~1 M params) on **RAVDESS** + **CREMA-D**.

* Features are extracted with the *same code the app runs* (`pipeline.affect`): audeering wav2vec2
  dimensional model for audio (hidden 1024 + A/D/V), HSEmotion `enet_b0_8_va_mtl` for face (embedding + V-A).
* Targets: emotion class, circumplex V-A per emotion, and a **mismatch** label from shuffled pairs
  (audio of one clip + face of a clip with a distant emotion).
* Output: `e3_head.pt` → copy to `models/fusion/e3_head.pt`. `fusion.py` then uses it automatically.

Runtime: feature extraction ~1-2 h on a T4 for RAVDESS + 2,000 CREMA-D clips; training ~10 min.

**Licences** – RAVDESS: CC BY-NC-SA 4.0 (Livingstone & Russo 2018). CREMA-D: ODbL (Cao et al. 2014). Cite both in the report.

In [ ]:
REPO = "https://github.com/<your-user>/vaktaai.git"   # private repo: use https://<token>@github.com/... or upload a zip
WORK = "/content/work"
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/vaktaai/e3"
!mkdir -p $WORK $OUT
!git clone -q $REPO /content/vaktaai || echo "repo already present"
%cd /content/vaktaai
# Colab already ships CUDA torch: install only what is missing (do NOT install requirements-ml.txt here).
!pip -q install mediapipe==0.10.14 hsemotion-onnx==0.3.1 transformers==4.46.3 praat-parselmouth soundfile imageio-ffmpeg pyyaml sqlmodel

In [ ]:
import os, sys, glob, random, json
os.environ["VAKTA_DATA_DIR"] = f"{WORK}/data"
os.environ["VAKTA_MODELS_DIR"] = "/content/vaktaai/models"
os.environ["VAKTA_MIN_DURATION_S"] = "1"
os.environ["VAKTA_ALLOW_DOWNLOADS"] = "1"
sys.path[:0] = ["/content/vaktaai/backend", "/content/vaktaai"]
import numpy as np, torch
SEED = 13
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

## 1 · Download datasets
RAVDESS audio-visual speech: 24 zips (~0.5 GB each) from Zenodo. Set `ACTORS` to a subset to save time.
CREMA-D: cloned with git-lfs (VideoFlash `.flv`, ~10 GB). `MAX_CREMA` caps how many clips are used.

In [ ]:
ACTORS = list(range(1, 25))
MAX_CREMA = 2000
for a in ACTORS:
    zip_path = f"{WORK}/ravdess_{a:02d}.zip"
    if not os.path.exists(zip_path):
        !wget -q -O $zip_path "https://zenodo.org/records/1188976/files/Video_Speech_Actor_{a:02d}.zip?download=1"
        !unzip -q -o $zip_path -d $WORK/ravdess
!apt-get -qq install git-lfs > /dev/null
!git lfs install --skip-repo
if not os.path.exists(f"{WORK}/CREMA-D"):
    !GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/CheyneyComputerScience/CREMA-D.git $WORK/CREMA-D
    !cd $WORK/CREMA-D && git lfs pull --include "VideoFlash/*"

In [ ]:
EMOTIONS = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
# Circumplex (valence, arousal) targets per category (Russell 1980; Warriner-style placements).
CIRCUMPLEX = {"neutral": (0.0, 0.0), "calm": (0.3, -0.5), "happy": (0.8, 0.5), "sad": (-0.7, -0.4),
              "angry": (-0.6, 0.8), "fearful": (-0.6, 0.6), "disgust": (-0.7, 0.3), "surprised": (0.3, 0.8)}
CREMA_MAP = {"NEU": "neutral", "HAP": "happy", "SAD": "sad", "ANG": "angry", "FEA": "fearful", "DIS": "disgust"}

clips = []
for path in glob.glob(f"{WORK}/ravdess/**/01-01-*.mp4", recursive=True):   # modality 01 = full AV
    parts = os.path.basename(path)[:-4].split("-")
    clips.append({"path": path, "dataset": "ravdess", "speaker": f"r{parts[6]}", "emotion": EMOTIONS[int(parts[2]) - 1]})
crema = sorted(glob.glob(f"{WORK}/CREMA-D/VideoFlash/*.flv"))
random.shuffle(crema)
for path in crema[:MAX_CREMA]:
    parts = os.path.basename(path)[:-4].split("_")
    if parts[2] in CREMA_MAP:
        clips.append({"path": path, "dataset": "crema", "speaker": f"c{parts[0]}", "emotion": CREMA_MAP[parts[2]]})
print(len(clips), "clips")

## 2 · Extract features with the app's own analysers
`analyse_file` runs ingest → face → face_va → voice → audio_va and leaves `audio_emb.npz` / `face_emb.npz`
in each session folder. Results are cached on Drive, so a disconnect only loses the current clip.

In [ ]:
from pathlib import Path
from pipeline.runner import analyse_file

feat_dir = Path(OUT) / "features"; feat_dir.mkdir(parents=True, exist_ok=True)
failed = 0
for i, c in enumerate(clips):
    key = f"{c['dataset']}_{Path(c['path']).stem}"
    target = feat_dir / f"{key}.npz"
    if target.exists():
        continue
    try:
        ctx = analyse_file(Path(c["path"]), ["face", "face_va", "voice", "audio_va"], session_id=key)
        a, f = np.load(ctx.path("audio_emb.npz")), np.load(ctx.path("face_emb.npz"))
        np.savez_compressed(target, audio=a["emb"], audio_va=a["va"], face=f["emb"], face_va=f["va"])
    except Exception as exc:
        failed += 1
        if failed < 10: print("skip", key, type(exc).__name__, exc)
    if i % 200 == 0: print(i, "/", len(clips))
print("failed:", failed)

In [ ]:
items = []
for c in clips:
    key = f"{c['dataset']}_{Path(c['path']).stem}"
    p = feat_dir / f"{key}.npz"
    if p.exists():
        d = np.load(p)
        if len(d["audio"]) and len(d["face"]):
            items.append({**c, "audio": d["audio"], "face": d["face"], "audio_va_raw": d["audio_va"].mean(0), "face_va_raw": d["face_va"].mean(0)})
print(len(items), "usable clips; dims", items[0]["audio"].shape[1], items[0]["face"].shape[1])

## 3 · Splits (speaker-independent) and mismatch pairs
Test = RAVDESS actors 19-24 + 15 % of CREMA-D speakers; validation = actors 17-18 + 10 % of CREMA-D speakers.

In [ ]:
crema_speakers = sorted({it["speaker"] for it in items if it["dataset"] == "crema"})
random.Random(SEED).shuffle(crema_speakers)
n = len(crema_speakers)
crema_test, crema_val = set(crema_speakers[: int(0.15 * n)]), set(crema_speakers[int(0.15 * n): int(0.25 * n)])
def split_of(it):
    if it["dataset"] == "ravdess":
        actor = int(it["speaker"][1:])
        return "test" if actor >= 19 else "val" if actor >= 17 else "train"
    return "test" if it["speaker"] in crema_test else "val" if it["speaker"] in crema_val else "train"
splits = {s: [it for it in items if split_of(it) == s] for s in ("train", "val", "test")}
print({k: len(v) for k, v in splits.items()})

train_a = np.concatenate([it["audio"] for it in splits["train"]]); train_f = np.concatenate([it["face"] for it in splits["train"]])
NORM = {"audio_mean": train_a.mean(0), "audio_std": train_a.std(0) + 1e-6, "face_mean": train_f.mean(0), "face_std": train_f.std(0) + 1e-6}

def distant(e1, e2):
    a, b = np.array(CIRCUMPLEX[e1]), np.array(CIRCUMPLEX[e2])
    return np.linalg.norm(a - b) > 0.8

def make_pairs(pool, mismatch_rate=0.5, seed=SEED):
    rng = random.Random(seed)
    pairs = []
    for it in pool:
        if rng.random() < mismatch_rate:
            other = rng.choice([o for o in pool if distant(it["emotion"], o["emotion"])])
            pairs.append((it, other, 1))
        else:
            pairs.append((it, it, 0))
    return pairs

MAX_LEN = 64
def collate(batch):
    A = max(len(a["audio"][:MAX_LEN]) for a, _, _ in batch); F = max(len(f["face"][:MAX_LEN]) for _, f, _ in batch)
    da, df = batch[0][0]["audio"].shape[1], batch[0][1]["face"].shape[1]
    audio, face = torch.zeros(len(batch), A, da), torch.zeros(len(batch), F, df)
    am, fm = torch.ones(len(batch), A, dtype=torch.bool), torch.ones(len(batch), F, dtype=torch.bool)
    emo, va, ava, fva, mis = [], [], [], [], []
    for i, (a, f, m) in enumerate(batch):
        xa = (a["audio"][:MAX_LEN] - NORM["audio_mean"]) / NORM["audio_std"]
        xf = (f["face"][:MAX_LEN] - NORM["face_mean"]) / NORM["face_std"]
        audio[i, :len(xa)] = torch.tensor(xa); am[i, :len(xa)] = False
        face[i, :len(xf)] = torch.tensor(xf); fm[i, :len(xf)] = False
        emo.append(EMOTIONS.index(a["emotion"])); va.append(CIRCUMPLEX[a["emotion"]])
        ava.append(CIRCUMPLEX[a["emotion"]]); fva.append(CIRCUMPLEX[f["emotion"]]); mis.append(m)
    t = lambda x, d=torch.float32: torch.tensor(x, dtype=d)
    return audio, face, am, fm, t(emo, torch.long), t(va), t(ava), t(fva), t(mis)

## 4 · Train

In [ ]:
from pipeline.nets import CrossAttentionFusionHead
import torch.nn.functional as F_

d_audio, d_face = items[0]["audio"].shape[1], items[0]["face"].shape[1]
model = CrossAttentionFusionHead(d_audio=d_audio, d_face=d_face, d_model=128, heads=4, layers=2,
                                 n_emotions=len(EMOTIONS), max_len=MAX_LEN).to(DEVICE)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M params")
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
val_pairs = make_pairs(splits["val"], seed=1); test_pairs = make_pairs(splits["test"], seed=2)

def run_epoch(pairs, train=True, bs=32):
    model.train(train)
    losses, outs = [], []
    order = random.sample(pairs, len(pairs)) if train else pairs
    for i in range(0, len(order), bs):
        audio, face, am, fm, emo, va, ava, fva, mis = [x.to(DEVICE) for x in collate(order[i:i + bs])]
        with torch.set_grad_enabled(train):
            out = model(audio, face, am, fm)
            congruent = mis == 0
            loss = (F_.mse_loss(out["audio_va"], ava) + F_.mse_loss(out["face_va"], fva)
                    + F_.binary_cross_entropy_with_logits(out["mismatch_logit"], mis))
            if congruent.any():
                loss = loss + F_.cross_entropy(out["emotion_logits"][congruent], emo[congruent]) \
                            + F_.mse_loss(out["va"][congruent], va[congruent])
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        outs.append({k: v.detach().cpu() for k, v in out.items()} | {"emo": emo.cpu(), "va_t": va.cpu(), "mis": mis.cpu()})
    return float(np.mean(losses)), outs

best, best_state = 1e9, None
for epoch in range(30):
    tr, _ = run_epoch(make_pairs(splits["train"], seed=100 + epoch), True)
    vl, _ = run_epoch(val_pairs, False)
    if vl < best: best, best_state = vl, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"epoch {epoch:02d} train {tr:.3f} val {vl:.3f}")
model.load_state_dict(best_state)

## 5 · Evaluate (learned vs rule-based distance baseline) and calibrate thresholds

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score
import yaml

def ccc(x, y):
    x, y = np.asarray(x), np.asarray(y)
    return 2 * np.cov(x, y)[0, 1] / (x.var() + y.var() + (x.mean() - y.mean()) ** 2)

def evaluate(pairs, threshold=None):
    _, outs = run_epoch(pairs, False)
    cat = lambda k: torch.cat([o[k] for o in outs]).numpy()
    mis, prob = cat("mis"), torch.sigmoid(torch.cat([o["mismatch_logit"] for o in outs])).numpy()
    cong = mis == 0
    emo_pred = cat("emotion_logits").argmax(1)
    va_pred, va_t = cat("va"), cat("va_t")
    if threshold is None:
        threshold = max(np.linspace(0.1, 0.9, 81), key=lambda t: f1_score(mis, prob >= t))
    # Rule baseline: raw per-modality V-A distance (what fusion.py uses without the head).
    raw_dist = np.array([np.linalg.norm(a["audio_va_raw"] - f["face_va_raw"]) for a, f, _ in pairs])
    return {"emotion_acc": float((emo_pred[cong] == cat("emo")[cong]).mean()),
            "ccc_valence": float(ccc(va_pred[cong, 0], va_t[cong, 0])), "ccc_arousal": float(ccc(va_pred[cong, 1], va_t[cong, 1])),
            "mismatch_auc_learned": float(roc_auc_score(mis, prob)), "mismatch_f1_learned": float(f1_score(mis, prob >= threshold)),
            "mismatch_auc_rule_distance": float(roc_auc_score(mis, raw_dist)), "threshold": float(threshold),
            "raw_distance_p95_congruent": float(np.percentile(raw_dist[cong], 95))}

val_metrics = evaluate(val_pairs)
rule_threshold = val_metrics["raw_distance_p95_congruent"]
test_metrics = evaluate(test_pairs, val_metrics["threshold"])
test_pairs_ravdess = [p for p in test_pairs if p[0]["dataset"] == "ravdess"]
ravdess_metrics = evaluate(test_pairs_ravdess, val_metrics["threshold"])
raw_test = np.array([np.linalg.norm(a["audio_va_raw"] - f["face_va_raw"]) for a, f, _ in test_pairs])
test_metrics["mismatch_f1_rule_distance"] = float(f1_score([m for *_, m in test_pairs], raw_test > rule_threshold))
print("val", json.dumps(val_metrics, indent=1)); print("test", json.dumps(test_metrics, indent=1))
print("RAVDESS held-out actors", json.dumps(ravdess_metrics, indent=1))
print(f"\nSet mismatch.distance_threshold in backend/pipeline/config/traits.yaml to ~{rule_threshold:.2f}")

In [ ]:
import csv
with open(f"{OUT}/e3_results.csv", "w", newline="") as fh:
    w = csv.writer(fh); w.writerow(["split", "metric", "value"])
    for split, m in (("test", test_metrics), ("ravdess_heldout_actors", ravdess_metrics)):
        for k, v in m.items(): w.writerow([split, k, v])
ckpt = {"config": model.config, "state_dict": model.state_dict(), "norm": NORM, "emotions": EMOTIONS,
        "mismatch_threshold": val_metrics["threshold"], "rule_distance_threshold": rule_threshold,
        "metrics": {"test": test_metrics, "ravdess": ravdess_metrics}, "seed": SEED}
torch.save(ckpt, f"{OUT}/e3_head.pt")
print("saved", f"{OUT}/e3_head.pt  ->  copy to models/fusion/e3_head.pt;  e3_results.csv -> eval/results/")